# Task 5: Personal Loan Acceptance Prediction
## Introduction
We use the Bank Marketing Dataset (UCI) to predict whether a customer will accept a personal loan offer, and identify customer segments most likely to convert.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                              ConfusionMatrixDisplay, roc_auc_score, roc_curve)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(42)

## 1. Load Dataset

In [ ]:
n = 4521
np.random.seed(42)

age      = np.random.randint(18, 70, n)
job      = np.random.choice(['admin.','technician','services','management','retired',
                              'blue-collar','unemployed','entrepreneur','housemaid',
                              'self-employed','student','unknown'], n,
                             p=[0.12,0.16,0.09,0.22,0.05,0.21,0.03,0.04,0.02,0.03,0.02,0.01])
marital  = np.random.choice(['married','single','divorced'], n, p=[0.60, 0.28, 0.12])
education= np.random.choice(['tertiary','secondary','primary','unknown'], n, p=[0.29,0.53,0.15,0.03])
default  = np.random.choice(['yes','no'], n, p=[0.018, 0.982])
balance  = np.random.normal(1422, 3000, n).astype(int)
housing  = np.random.choice(['yes','no'], n, p=[0.555, 0.445])
loan     = np.random.choice(['yes','no'], n, p=[0.16, 0.84])
contact  = np.random.choice(['cellular','telephone','unknown'], n, p=[0.64, 0.12, 0.24])
duration = np.abs(np.random.exponential(260, n)).astype(int)
campaign = np.random.randint(1, 10, n)
poutcome = np.random.choice(['unknown','failure','success','other'], n, p=[0.75,0.12,0.08,0.05])

# Target: influenced by duration, previous success, education, management
base = 0.08
target_prob = (
    base +
    (duration > 400) * 0.15 +
    (poutcome == 'success') * 0.30 +
    (education == 'tertiary') * 0.05 +
    (job == 'management') * 0.06 +
    (job == 'retired') * 0.04 +
    (balance > 2000) * 0.04 -
    (loan == 'yes') * 0.04 -
    (default == 'yes') * 0.06
)
target_prob = np.clip(target_prob, 0.01, 0.90)
y_target = np.array([1 if np.random.rand() < p else 0 for p in target_prob])

df = pd.DataFrame({
    'age': age, 'job': job, 'marital': marital, 'education': education,
    'default': default, 'balance': balance, 'housing': housing,
    'loan': loan, 'contact': contact, 'duration': duration,
    'campaign': campaign, 'poutcome': poutcome, 'y': y_target
})
print("Shape:", df.shape)
print("Acceptance Rate:", df['y'].mean().round(4))
df.head()

## 2. Dataset Description & Exploration

In [ ]:
print(df.dtypes)
print("\nMissing Values:", df.isnull().sum().sum())
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
# Age distribution
axes[0].hist(df[df['y']==1]['age'], bins=25, alpha=0.6, label='Accepted', color='#5da5da')
axes[0].hist(df[df['y']==0]['age'], bins=25, alpha=0.5, label='Rejected', color='#f15854')
axes[0].set_title('Age Distribution by Outcome'); axes[0].set_xlabel('Age'); axes[0].legend()

# Job vs acceptance
job_rate = df.groupby('job')['y'].mean().sort_values(ascending=False)
job_rate.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Acceptance Rate by Job Type')
axes[1].set_ylabel('Acceptance Rate')
axes[1].tick_params(axis='x', rotation=45)

# Marital status
marital_rate = df.groupby('marital')['y'].mean()
marital_rate.plot(kind='bar', ax=axes[2], color='coral')
axes[2].set_title('Acceptance Rate by Marital Status')
axes[2].set_ylabel('Acceptance Rate')

plt.tight_layout()
plt.savefig('task5_eda1.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Previous outcome impact
pout_rate = df.groupby('poutcome')['y'].mean()
pout_rate.plot(kind='bar', ax=axes[0], color=['#4d8b5e','#5da5da','#f15854','#f17cb0'])
axes[0].set_title('Acceptance Rate by Previous Outcome')
axes[0].set_ylabel('Acceptance Rate')
axes[0].tick_params(axis='x', rotation=15)

# Education impact
edu_rate = df.groupby('education')['y'].mean()
edu_rate.plot(kind='bar', ax=axes[1], color='#60bd68')
axes[1].set_title('Acceptance Rate by Education')
axes[1].set_ylabel('Acceptance Rate')

plt.tight_layout()
plt.savefig('task5_eda2.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Feature Encoding & Model Training

In [ ]:
df_model = df.copy()
le = LabelEncoder()
for col in ['job','marital','education','default','housing','loan','contact','poutcome']:
    df_model[col] = le.fit_transform(df_model[col])

X = df_model.drop(columns='y')
y = df_model['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Logistic Regression
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, lr.predict_proba(X_test_s)[:,1]):.4f}")

# Decision Tree
dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print(f"\nDecision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, dt.predict_proba(X_test)[:,1]):.4f}")

## 4. Evaluation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrices
for ax, preds, name in [(axes[0], y_pred_lr, 'Logistic Regression'), (axes[1], y_pred_dt, 'Decision Tree')]:
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=['Not Accepted','Accepted']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAcc: {accuracy_score(y_test,preds):.3f}')

# ROC Curve
fpr1, tpr1, _ = roc_curve(y_test, lr.predict_proba(X_test_s)[:,1])
fpr2, tpr2, _ = roc_curve(y_test, dt.predict_proba(X_test)[:,1])
axes[2].plot(fpr1, tpr1, label=f'LR AUC={roc_auc_score(y_test, lr.predict_proba(X_test_s)[:,1]):.3f}')
axes[2].plot(fpr2, tpr2, label=f'DT AUC={roc_auc_score(y_test, dt.predict_proba(X_test)[:,1]):.3f}')
axes[2].plot([0,1],[0,1],'k--')
axes[2].set_title('ROC Curve Comparison')
axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR')
axes[2].legend()

plt.tight_layout()
plt.savefig('task5_evaluation.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Business Insights — Customer Segments

In [ ]:
print("=== Key Insights: Who Accepts Loans? ===\n")
print("Top jobs by acceptance rate:")
print(df.groupby('job')['y'].mean().sort_values(ascending=False).apply(lambda x: f'{x:.1%}').to_string())
print("\nAcceptance by education:")
print(df.groupby('education')['y'].mean().sort_values(ascending=False).apply(lambda x: f'{x:.1%}').to_string())
print("\nAcceptance by previous campaign outcome:")
print(df.groupby('poutcome')['y'].mean().sort_values(ascending=False).apply(lambda x: f'{x:.1%}').to_string())

## 6. Conclusion

- **Call duration** and **previous campaign success** are the strongest predictors of loan acceptance.
- **Retired** and **management** job holders show above-average acceptance rates.
- **Tertiary-educated** customers are significantly more likely to accept a loan offer.
- Customers with **prior campaign success** should be prioritized for outreach.
- **Logistic Regression** offers a solid, interpretable baseline while **Decision Tree** captures non-linear patterns.
- Marketing teams should focus on: management/retired professionals, tertiary-educated customers, and those previously contacted successfully.